# Lab 1 — Version 1

## Approach
Predict whether a LinkedIn post performs below, typical or above its author's usual engagement.

This version extends the official gradient boosting baseline with recent author-history and writing-style features, then combines it with a TF-IDF text classifier.

## Data and validation
The official train/test split is preserved: 71,395 training posts and 17,623 test posts.
Within train, 20% of authors are reserved for validation using GroupShuffleSplit with seed 42. Training and validation authors do not overlap.

Models and the TF-IDF vocabulary are fitted on the fitting portion only. Configuration selection uses validation macro-F1; the selected models are then refitted on the full training set.

## Method
- **History and style:** reaction averages over the previous 3, 5 and 10 posts, recent variability and trend, text length per word, exclamation count and uppercase ratio.
- **Text:** word unigram/bigram TF-IDF with balanced logistic regression.
- **Combination:** 70% enriched tabular model and 30% text classifier. The typical-class score is multiplied by 1.4 before selecting the class.

Historical features exclude the current post. Current-post reactions, residuals and labels are not model inputs.

## Results
| Validation variant | Macro-F1 |
|---|---:|
| Official baseline | 0.5546 |
| History/style model | 0.5740 |
| Text only | 0.4466 |
| History/style + adjusted class decision | 0.6112 |
| Selected combination | **0.6176** |

The text model contributes complementary information, but part of the improvement also comes from adjusting the class decision rule.

| Model on official test | Macro-F1 | QWK | Accuracy |
|---|---:|---:|---:|
| Official baseline | 0.5410 | 0.4576 | 0.5465 |
| Version 1 | **0.6052** | **0.5240** | **0.6444** |

The gain is concentrated in typical and above; below-class F1 is almost unchanged. All 17,623 test predictions were verified with the official evaluate.py.

## Limitations
Configuration selection uses a single validation split. Historical engagement, follower counts and age-at-crawl features retain the dataset's snapshot limitations and do not fully reconstruct information available at publication time.

## Reproduction
From the repository root, with requirements.txt installed:

```bash
python improved.py
python evaluate.py --predictions lab1_submission/predictions.csv
```

Predictions, metrics and validation selection are saved in lab1_submission/.
The code cell below displays saved test results; it does not retrain the models.


In [ ]:
import json
from pathlib import Path
report = json.loads(Path("lab1_submission/metrics.json").read_text())
report["test"]
